In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml
import spacy
from corextopic import corextopic as ct
from dvclive import Live
from matplotlib.figure import Figure
from spacy.tokens import DocBin

from job_post_nlp.utils.interactive import try_inter

try_inter()
from job_post_nlp.prepare import corpus_unpack, register_extensions, load_data,register_extensions, load_texts # noqa: E402
from job_post_nlp.utils.find_project_root import find_project_root  # noqa: E402
from job_post_nlp.evaluate import load_model  # noqa: E402
from job_post_nlp.train import load_corpus_split, load_tdm  # noqa: E402
import helpfuncs as hf

/home/b281467@PROD.SITAD.DK/.conda/envs/jobpostnlp/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Process
data = hf.load_everything()
tdm = data['tdm']
tdm_info = data['tdm_info']


In [3]:
hf.gram_statistics(data)

Total documents: 1876989, Total tokens: 482044

Statistics for 1-grams:
Total unique 1-grams: 439138
Total unique ngrams with at least 2 occurrences: 439138
Total unique ngrams with at least 3 occurrences: 439138
Total unique ngrams with at least 4 occurrences: 369433
Total unique ngrams with at least 5 occurrences: 324232
Total unique ngrams with at least 938 documents (0.05%): 12650
Total unique ngrams with at least 1876 documents (0.1%): 8173
Total unique ngrams with at least 9384 documents (0.5%): 2812
Total unique ngrams with at least 18769 documents (1.0%): 1773

Statistics for 2-grams:
Total unique 2-grams: 30806
Total unique ngrams with at least 2 occurrences: 30806
Total unique ngrams with at least 3 occurrences: 30806
Total unique ngrams with at least 4 occurrences: 30806
Total unique ngrams with at least 5 occurrences: 30806
Total unique ngrams with at least 938 documents (0.05%): 30806
Total unique ngrams with at least 1876 documents (0.1%): 15161
Total unique ngrams with a

In [6]:
anchors = hf.read_anchors_from_yaml('note.yaml')

In [7]:
anchors

[['fleksibel',
  'fleksibilitet',
  'fleksibel arbejdstid',
  'fleksibel arbejdstid',
  'flekstid',
  'flekstidsordning',
  'flextid',
  'flextidsordning',
  'fixtid',
  'fikstid'],
 ['familievenlig', 'familievenlige', 'familieliv'],
 ['hjemmearbejde', 'hjemmearbejdsplads', 'hjemmearbejdsdag', 'hjemmefra'],
 ['vagtskema', 'vagtplan', 'spidsbelastning'],
 ['natarbejde',
  'weekend',
  'weekendarbejde',
  'helligdage',
  'nattevagt',
  'nattevagte',
  'nattevagterne',
  'aften'],
 ['vagtplan'],
 ['ambitiøs',
  'ambition',
  'udvikling',
  'karriere',
  'udviklingsmuligheder',
  'udviklingsplaner',
  'udviklingsforløb',
  'udviklingsprogram'],
 ['faglig', 'faglighed', 'sparring'],
 ['stillingen',
  'samtaler',
  'snarest muligt',
  'mulig',
  'ansøgning',
  'ringe',
  'ansøgning mail',
  'ansøgning',
  'sende',
  'cv',
  'sende ansøgning',
  'ansøgning cv',
  'send',
  'send ansøgning',
  'ansøgning sende',
  'modtage ansøgning'],
 ['køn',
  'etnisk',
  'religion',
  'religion etnisk',
  

In [8]:
# Print most common tokens in tdm
freq = tdm.sum(axis=0).A1
# Sort tokens by frequency
freq_sort = np.argsort(freq)[::-1]
freq_sorted = freq[freq_sort]

In [9]:
sorted_tokens = np.array(tdm_info['vocab'] )[freq_sort]
sorted_tokens

array(['søge', 'arbejde', 'god', ..., 'aabylæs', 'aabrandt',
       'forlystelser'], shape=(482044,), dtype='<U58')

In [10]:
topn = 1000
print(f"Top {topn} tokens by frequency:")
for token, frequency in zip(sorted_tokens[:topn], freq_sorted[:topn]):
    print(f"{token}: {frequency}")

Top 1000 tokens by frequency:
søge: 1531923
arbejde: 1490004
god: 1171709
stilling: 1169962
erfaring: 1008833
samt: 1007670
samarbejde: 938583
tilbyde: 930904
opgave: 903639
ansøgning: 880115
del: 875645
stor: 866371
mulighed: 862761
forvente: 814158
medarbejder: 808202
udvikling: 791042
både: 761137
faglig: 751767
time: 741720
job: 734931
gerne: 662382
ny: 650198
ikke: 647745
uge: 633926
sende: 624324
høj: 622681
hverdag: 621247
få: 620518
kollega: 614649
team: 609795
se: 596817
fokus: 594253
løn: 590805
overenskomst: 576767
lyst: 572310
spændende: 564698
år: 561468
inden: 558100
skabe: 542838
tage: 533331
indgå: 518454
ansvar: 514498
tæt: 512861
ønske: 512266
selvstændig: 506807
bestå: 504584
udvikle: 503000
hos: 499476
arbejdsplads: 484457
kontakte: 483427
dag: 477031
gælde: 472868


din: 470819
kollegaa: 463079
fleksibel: 462528
uddannelse: 460475
løbende: 446357
velkommen: 443992
cv: 435850
afdeling: 433514
dygtig: 432842
forskellig: 431699
ansættelse: 426637
kommune: 420597
relevant: 419801
vigtig: 417658
tale: 416958
arbejdstid: 409557
dansk: 406560
område: 405950
arbejdsmiljø: 401910
fast: 399228
tid: 394862
ansætte: 394728
sætte: 392977
positiv: 388185
gælde overenskomst: 385547
afholde: 384880
oplysning: 382446
personlig: 379014
mulig: 377706
hel: 376974
daglig: 375333
ansættelsesvilkår: 374929
sikre: 371935
virksomhed: 368757
læse: 366624
ansøgningsfrist: 364151
forhold: 360498
give: 360023
kontakt: 356429
indenfor: 355952
spørgsmål: 350565
engagere: 348395
når: 347215
bidrage: 345632
aftale: 343413
uddanne: 340838
kvalitet: 340306
herunder: 338512
kunde: 334363
kendskab: 330075
varetage: 327626
gå: 326923
borger: 324946
stærk: 323847
følge: 323729
behov: 321613
komme: 321434
to: 320740
overblik: 320698
samtale: 319741
senest: 318741
velkommen kontakte: 317

In [11]:
topn = 1000
print(f"Bottom {topn} tokens by frequency:")
for token, frequency in zip(sorted_tokens[-topn:], freq_sorted[-topn:]):
    print(f"{token}: {frequency}")

Bottom 1000 tokens by frequency:
motorcykelsæde: 3
motorcykelklub: 3
motorbeskyttelse: 3
musiksysteme: 3
musikstudiemedarbejd: 3
ønse: 3
ønning: 3
ønkses: 3
forhandlingsansvar: 3
forhandlingplacering: 3
forhandlignsberettiget: 3
forhandleruddannelse: 3
forhandlerstol: 3
forhandlerservice: 3
spillereglerne: 3
spillerderfor: 3
motorikug: 3
motoriktræning: 3
motoriktilbud: 3
motorikprojekter: 3
musikskoleadministration: 3
musikskoel: 3
musikshows: 3
musikschulen: 3
motorikdag: 3
motorikcafé: 3
musikskoleområdum: 3
musikskoleniveaue: 3
musikskolelærerlignende: 3
motorisering: 3
motorikvejlede: 3
motorikuge: 3
musikredskab: 3
musikredaktør: 3
musikpædagoger: 3
musikpædagogen: 3
motorikplads: 3
motorikmedarbejder: 3
motoriklokal: 3
motorikhal: 3
musikpolitik: 3
musikorganisation: 3
musikområder: 3
motorlær: 3
motorkørertøjer: 3
motorkædesav: 3
ønskescenarium: 3
ønskertil: 3
ønskeom: 3
ønskemad: 3
adademisk: 3
adacta: 3
acut: 3
acum: 3
ønsketime: 3
musikproducenten: 3
specialuddannete: 3
spec


specialundervisningselev: 3
formidlingssysteme: 3
motorregistrering: 3
musikog: 3
musiknørder: 3
musikmuseets: 3
musikmuse: 3
musikmaterialesamling: 3
musikmagne: 3
ørekirurgiske: 3
actionlog: 3
actigraph: 3
actavis: 3
motorsportscent: 3
motorskib: 3
motorsavsmekaniker: 3
motorsag: 3
motorrummet: 3
musikpsykoterapeutisk: 3
musikprofilklasse: 3
øntgen: 3
ønsøg: 3
ønsklig: 3
ønskeværdig: 3
actiumplus: 3
actionrum: 3
motorsystem: 3
motorsyste: 3
motorsvejsafkørsel: 3
motorsportsfacilitet: 3
acoa: 3
acitestapning: 3
acitespunktur: 3
acinta: 3
musiklektionerne: 3
musiklejr: 3
musiklegeplads: 3
musikledsagelsen: 3
musikland: 3
musikkunstner: 3
musikkundskaber: 3
musikkulturel: 3
formidlingszone: 3
formidlingsworkshops: 3
formidlingsvideo: 3
specialundervisningspædagogik: 3
specialundervisningspersonalet: 3
forhandlingsberettigedefaglig: 3
forhandlingsberetiget: 3
acontofakturering: 3
formidlingsstyrke: 3
formidlingsstil: 3
formidlingsspecialist: 3
formidlingssekretariat: 3
formidlingssatsni

In [12]:
freq_sorted[(sorted_tokens =='god overblik')]

array([53165])

In [13]:
freq.shape

(482044,)

In [29]:
(freq==1).sum(), (freq==2).sum(), (freq==3).sum(), (freq==4).sum()

(np.int64(0), np.int64(129021), np.int64(69705), np.int64(45201))

In [14]:
hf.check_anchors(anchors,data)


Checking anchor: fleksibel
'fleksibel' occurs 462528 times in the TDM.
'fleksibilitet' occurs 104468 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'flekstid' occurs 11214 times in the TDM.
'flekstidsordning' occurs 5758 times in the TDM.
'flextid' occurs 6775 times in the TDM.
'flextidsordning' occurs 1328 times in the TDM.
'fixtid' occurs 124 times in the TDM.
'fikstid' occurs 22 times in the TDM.

Checking anchor: familievenlig
'familievenlig' occurs 4954 times in the TDM.
'familievenlige' occurs 101 times in the TDM.
'familieliv' occurs 6870 times in the TDM.

Checking anchor: hjemmearbejde
'hjemmearbejde' occurs 5195 times in the TDM.
'hjemmearbejdsplads' occurs 2437 times in the TDM.
'hjemmearbejdsdag' occurs 4105 times in the TDM.
'hjemmefra' occurs 9532 times in the TDM.

Checking anchor: vagtskema
'vagtskema' occurs 1581 times in the TDM.
'vagtplan' occurs 38305 times in the TDM.
'spidsbelastning' 

In [15]:
hf.find_similar_words('fleksib',data)

Found 402 similar words, showing top 250.
Similar words to 'fleksib':
'fleksibel' occurs 462528 times in the TDM.
'fleksibilitet' occurs 104468 times in the TDM.
'fleksibel arbejdstid' occurs 32134 times in the TDM.
'fleksibel forhold' occurs 28457 times in the TDM.
'mødestabil fleksibel' occurs 19839 times in the TDM.
'fleksibelt' occurs 19622 times in the TDM.
'stabil fleksibel' occurs 17202 times in the TDM.
'fleksibel stabil' occurs 17081 times in the TDM.
'engagere fleksibel' occurs 16718 times in the TDM.
'stor fleksibilitet' occurs 16112 times in the TDM.
'fleksibel omstillingsparat' occurs 15421 times in the TDM.
'fleksible' occurs 13850 times in the TDM.
'fleksibilit' occurs 13266 times in the TDM.
'fleksibel arbejde' occurs 12903 times in the TDM.
'fleksibel god' occurs 11386 times in the TDM.
'selvstændig fleksibel' occurs 10733 times in the TDM.
'fleksibel mødestabil' occurs 10522 times in the TDM.
'fleksibel arbejdsplads' occurs 10377 times in the TDM.
'fleksibel positiv' 

In [14]:
hf.find_similar_words('flekstid',data)

Similar words to 'flekstid':
'flekstid' occurs 11214 times in the TDM.
'flekstidsordning' occurs 5758 times in the TDM.
'flekstidsaftale' occurs 187 times in the TDM.
'flekstidsaftal' occurs 118 times in the TDM.
'flekstidssystem' occurs 22 times in the TDM.
'flekstidordning' occurs 8 times in the TDM.
'flekstider' occurs 7 times in the TDM.
'flekstidsadministratione' occurs 6 times in the TDM.
'flekstidsordninger' occurs 6 times in the TDM.
'flekstidsregler' occurs 6 times in the TDM.
'flekstidsordni' occurs 4 times in the TDM.
'harflekstidsordning' occurs 4 times in the TDM.
'flekstidsadministration' occurs 3 times in the TDM.
'flekstidsansatte' occurs 3 times in the TDM.
'flekstidso' occurs 3 times in the TDM.
'flekstidsskema' occurs 3 times in the TDM.
'harflekstid' occurs 3 times in the TDM.
'erflekstid' occurs 2 times in the TDM.
'flekstiden' occurs 2 times in the TDM.
'flekstidsløsning' occurs 2 times in the TDM.
'flekstidsopfølgning' occurs 2 times in the TDM.
'flekstidsregnska

In [15]:
hf.find_similar_words('hjemmefra',data)

Similar words to 'hjemmefra':
'hjemmefra' occurs 9532 times in the TDM.
'hjemmefradu' occurs 37 times in the TDM.
'akuthjemmefra' occurs 2 times in the TDM.


In [16]:
hf.find_similar_words('ministeri',data)

Similar words to 'ministeri':
'finansministeri' occurs 67003 times in the TDM.
'overenskomst finansministeri' occurs 29408 times in the TDM.
'forsvarsministeri' occurs 20189 times in the TDM.
'ministeri' occurs 10956 times in the TDM.
'ministerium' occurs 9718 times in the TDM.
'kirkeministerium' occurs 7001 times in the TDM.
'forsvarsministeriets' occurs 6390 times in the TDM.
'skatteministeriet' occurs 6344 times in the TDM.
'skatteministeri' occurs 6018 times in the TDM.
'skatteministeriets' occurs 4244 times in the TDM.
'undervisningsministeriets' occurs 2964 times in the TDM.
'fødevareministeri' occurs 2730 times in the TDM.
'udenrigsministeriet' occurs 2394 times in the TDM.
'forskningsministeri' occurs 2328 times in the TDM.
'kulturministerium' occurs 1840 times in the TDM.
'boligministeriet' occurs 1543 times in the TDM.
'undervisningsministeriet' occurs 1524 times in the TDM.
'miljøministerium' occurs 1442 times in the TDM.
'udenrigsministeriets' occurs 1358 times in the TDM.
